# Proyecto #2 - Data Mining (PBL)
## Semana 3: Validacion, Optimizacion y Comparacion Final

---

## 1. Importacion de librerias

In [2]:
%pip install matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 8.3 MB/s eta 0:00:01
   ------------ --------------------------- 2.6/8.1 MB 9.9 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.1 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 11.6 MB/s  0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------- ---------------------

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, make_scorer

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RANDOM_STATE = 42
TEST_SIZE = 0.20
LABELS_CLASE = [2, 4]

print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


## 2. Confirmacion del dataset final y split heredado

In [4]:
candidatos = [
    Path('data/Datos_modelado.csv'),
    Path('../data/Datos_modelado.csv'),
    Path('Datos_modelado.csv')
]
ruta_datos = next((p for p in candidatos if p.exists()), None)
if ruta_datos is None:
    raise FileNotFoundError('No se encontro Datos_modelado.csv en rutas esperadas.')

df = pd.read_csv(ruta_datos)
df.columns = df.columns.str.strip()

expected_rows = 675
expected_cols = 10

print('=' * 90)
print('VALIDACION DE INSUMOS DE SEMANA 2')
print('=' * 90)
print(f'Archivo detectado: {ruta_datos}')
print(f'Dimension actual : {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Esperado S2      : {expected_rows} filas x {expected_cols} columnas')
print(f'Coincide?        : {df.shape == (expected_rows, expected_cols)}')
print('\nDistribucion de clases:')
print(df['Class'].value_counts().sort_index())
print('=' * 90)

VALIDACION DE INSUMOS DE SEMANA 2
Archivo detectado: ..\data\Datos_modelado.csv
Dimension actual : 675 filas x 10 columnas
Esperado S2      : 675 filas x 10 columnas
Coincide?        : True

Distribucion de clases:
Class
2    439
4    236
Name: count, dtype: int64


In [5]:
target_col = 'Class'
id_col = 'Sample code number'

if id_col in df.columns:
    feature_cols = [c for c in df.columns if c not in [id_col, target_col]]
else:
    feature_cols = [c for c in df.columns if c != target_col]

X_all = df[feature_cols].values
y = df[target_col].values

top3_features = ['Bare Nuclei', 'Uniformity of Cell Size', 'Uniformity of Cell Shape']
if not set(top3_features).issubset(set(feature_cols)):
    raise ValueError('No se encontraron todas las variables del escenario Top-3.')

X_top = df[top3_features].values

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

X_train_top, X_test_top, _, _ = train_test_split(
    X_top, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

escenarios = {
    'A - Todas las variables': (X_train_all, X_test_all),
    'B - Top 3 variables': (X_train_top, X_test_top),
}

def ratio_clases(arr):
    s = pd.Series(arr).value_counts(normalize=True).sort_index()
    return s

print('=' * 90)
print('CONFIRMACION DEL SPLIT (S2 -> S3)')
print('=' * 90)
print(f'X_train_all: {X_train_all.shape} | X_test_all: {X_test_all.shape}')
print(f'X_train_top: {X_train_top.shape} | X_test_top: {X_test_top.shape}')
print(f'y_train    : {y_train.shape}      | y_test    : {y_test.shape}')

dist_total = ratio_clases(y)
dist_train = ratio_clases(y_train)
dist_test = ratio_clases(y_test)

tabla_dist = pd.DataFrame({
    'Total': dist_total,
    'Train': dist_train,
    'Test': dist_test
}).fillna(0).round(4)

print('\nProporcion de clases (2=Benigno, 4=Maligno):')
print(tabla_dist)
print('=' * 90)

CONFIRMACION DEL SPLIT (S2 -> S3)
X_train_all: (540, 9) | X_test_all: (135, 9)
X_train_top: (540, 3) | X_test_top: (135, 3)
y_train    : (540,)      | y_test    : (135,)

Proporcion de clases (2=Benigno, 4=Maligno):
    Total  Train    Test
2  0.6504   0.65  0.6519
4  0.3496   0.35  0.3481


## 3. Metodologia de validacion

**Esquema adoptado: 5-Fold Stratified Cross-Validation**

Criterios del diseno experimental:
1. Mantener proporcion de clases en cada fold.
2. Evaluar todos los modelos bajo condiciones identicas.
3. Reportar media y desviacion estandar de metricas por modelo y escenario.
4. Mantener el test set virgen para la verificacion final.

In [6]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision_weighted',
    'recall_macro': 'recall_macro',
    'f1': 'f1_weighted',
    'roc_auc': 'roc_auc_ovr_weighted'
}

modelos_base = {
    'Regresion Logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    'KNN (k=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE, probability=True))
    ])
}

print('=' * 90)
print('CONFIGURACION DE VALIDACION')
print('=' * 90)
print('Estrategia: 5-Fold Stratified CV')
print(f'Folds: {cv_strategy.get_n_splits()}')
print(f'Metricas: {list(scoring_metrics.keys())}')
print(f'Modelos: {list(modelos_base.keys())}')
print('=' * 90)

CONFIGURACION DE VALIDACION
Estrategia: 5-Fold Stratified CV
Folds: 5
Metricas: ['accuracy', 'precision', 'recall_macro', 'f1', 'roc_auc']
Modelos: ['Regresion Logistica', 'KNN (k=5)', 'SVM (RBF)']


## 4. Ejecucion de validacion base (pre-tuning)

In [7]:
cv_results_store = {}

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    print('=' * 90)
    print(f'VALIDACION CRUZADA | ESCENARIO: {nombre_escenario}')
    print('=' * 90)

    for modelo_nombre, modelo_pipeline in modelos_base.items():
        print(f'\n>>> {modelo_nombre} | {nombre_escenario}')

        cv_results = cross_validate(
            modelo_pipeline,
            X_tr_esc,
            y_train,
            cv=cv_strategy,
            scoring=scoring_metrics,
            n_jobs=-1,
            return_train_score=False
        )

        key = f'{modelo_nombre} | {nombre_escenario}'
        cv_results_store[key] = cv_results

        for metrica in ['accuracy', 'precision', 'recall_macro', 'f1', 'roc_auc']:
            vals = cv_results[f'test_{metrica}']
            print(f'   {metrica.upper():15}: {vals.mean():.4f} +/- {vals.std():.4f}')

print('\n' + '=' * 90)
print('VALIDACION BASE COMPLETADA')
print('=' * 90)

VALIDACION CRUZADA | ESCENARIO: A - Todas las variables

>>> Regresion Logistica | A - Todas las variables
   ACCURACY       : 0.9704 +/- 0.0283
   PRECISION      : 0.9705 +/- 0.0284
   RECALL_MACRO   : 0.9664 +/- 0.0351
   F1             : 0.9702 +/- 0.0287
   ROC_AUC        : 0.9941 +/- 0.0074

>>> KNN (k=5) | A - Todas las variables
   ACCURACY       : 0.9685 +/- 0.0224
   PRECISION      : 0.9685 +/- 0.0225
   RECALL_MACRO   : 0.9636 +/- 0.0272
   F1             : 0.9684 +/- 0.0226
   ROC_AUC        : 0.9843 +/- 0.0262

>>> SVM (RBF) | A - Todas las variables
   ACCURACY       : 0.9611 +/- 0.0312
   PRECISION      : 0.9624 +/- 0.0310
   RECALL_MACRO   : 0.9617 +/- 0.0341
   F1             : 0.9613 +/- 0.0312
   ROC_AUC        : 0.9898 +/- 0.0109
VALIDACION CRUZADA | ESCENARIO: B - Top 3 variables

>>> Regresion Logistica | B - Top 3 variables
   ACCURACY       : 0.9648 +/- 0.0271
   PRECISION      : 0.9648 +/- 0.0272
   RECALL_MACRO   : 0.9583 +/- 0.0305
   F1             : 0.9647 +

## 5. Tablas de resultados de cross-validation

In [6]:
rows_cv = []

for key, results in cv_results_store.items():
    modelo_nombre, escenario = key.split(' | ')
    rows_cv.append({
        'Modelo': modelo_nombre,
        'Escenario': escenario,
        'Accuracy (mean+/-std)': f"{results['test_accuracy'].mean():.4f} +/- {results['test_accuracy'].std():.4f}",
        'Precision (mean+/-std)': f"{results['test_precision'].mean():.4f} +/- {results['test_precision'].std():.4f}",
        'Recall (mean+/-std)': f"{results['test_recall_macro'].mean():.4f} +/- {results['test_recall_macro'].std():.4f}",
        'F1 (mean+/-std)': f"{results['test_f1'].mean():.4f} +/- {results['test_f1'].std():.4f}",
        'AUC (mean+/-std)': f"{results['test_roc_auc'].mean():.4f} +/- {results['test_roc_auc'].std():.4f}"
    })

df_cv_results = pd.DataFrame(rows_cv)

print('=' * 140)
print('TABLA GLOBAL: RESULTADOS 5-FOLD STRATIFIED CV')
print('=' * 140)
print(df_cv_results.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA POR ESCENARIO A - TODAS LAS VARIABLES')
print('=' * 140)
print(df_cv_results[df_cv_results['Escenario'] == 'A - Todas las variables'].drop(columns=['Escenario']).to_string(index=False))

print('\n' + '=' * 140)
print('TABLA POR ESCENARIO B - TOP 3 VARIABLES')
print('=' * 140)
print(df_cv_results[df_cv_results['Escenario'] == 'B - Top 3 variables'].drop(columns=['Escenario']).to_string(index=False))

TABLA GLOBAL: RESULTADOS 5-FOLD STRATIFIED CV
             Modelo               Escenario Accuracy (mean+/-std) Precision (mean+/-std) Recall (mean+/-std)   F1 (mean+/-std)  AUC (mean+/-std)
Regresion Logistica A - Todas las variables     0.9704 +/- 0.0283      0.9705 +/- 0.0284   0.9664 +/- 0.0351 0.9702 +/- 0.0287 0.9941 +/- 0.0074
          KNN (k=5) A - Todas las variables     0.9685 +/- 0.0224      0.9685 +/- 0.0225   0.9636 +/- 0.0272 0.9684 +/- 0.0226 0.9843 +/- 0.0262
          SVM (RBF) A - Todas las variables     0.9611 +/- 0.0312      0.9624 +/- 0.0310   0.9617 +/- 0.0341 0.9613 +/- 0.0312 0.9898 +/- 0.0109
Regresion Logistica     B - Top 3 variables     0.9648 +/- 0.0271      0.9648 +/- 0.0272   0.9583 +/- 0.0305 0.9647 +/- 0.0272 0.9912 +/- 0.0112
          KNN (k=5)     B - Top 3 variables     0.9667 +/- 0.0239      0.9671 +/- 0.0234   0.9634 +/- 0.0246 0.9667 +/- 0.0237 0.9823 +/- 0.0167
          SVM (RBF)     B - Top 3 variables     0.9574 +/- 0.0319      0.9577 +/- 0.

In [7]:
# Comparacion CV vs Test para confirmar consistencia de generalizacion
test_rows = []

for nombre_esc, (X_tr_esc, X_te_esc) in escenarios.items():
    for modelo_nombre, modelo_pipeline in modelos_base.items():
        modelo_pipeline.fit(X_tr_esc, y_train)
        y_pred = modelo_pipeline.predict(X_te_esc)
        y_proba = modelo_pipeline.predict_proba(X_te_esc)[:, 1]

        acc_test = accuracy_score(y_test, y_pred)
        rec_test = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1_test = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        auc_test = roc_auc_score(y_test, y_proba, labels=LABELS_CLASE)

        key = f'{modelo_nombre} | {nombre_esc}'
        acc_cv = cv_results_store[key]['test_accuracy'].mean()
        rec_cv = cv_results_store[key]['test_recall_macro'].mean()
        f1_cv = cv_results_store[key]['test_f1'].mean()
        auc_cv = cv_results_store[key]['test_roc_auc'].mean()

        test_rows.append({
            'Modelo': modelo_nombre,
            'Escenario': nombre_esc,
            'Acc_CV': round(acc_cv, 4),
            'Acc_Test': round(acc_test, 4),
            'Gap_Acc': round(acc_test - acc_cv, 4),
            'Recall_CV': round(rec_cv, 4),
            'Recall_Test': round(rec_test, 4),
            'Gap_Recall': round(rec_test - rec_cv, 4),
            'F1_CV': round(f1_cv, 4),
            'F1_Test': round(f1_test, 4),
            'AUC_CV': round(auc_cv, 4),
            'AUC_Test': round(auc_test, 4)
        })

df_cv_vs_test = pd.DataFrame(test_rows)
print('=' * 140)
print('TABLA DE CONSISTENCIA: CV VS TEST')
print('=' * 140)
print(df_cv_vs_test.to_string(index=False))

TABLA DE CONSISTENCIA: CV VS TEST
             Modelo               Escenario  Acc_CV  Acc_Test  Gap_Acc  Recall_CV  Recall_Test  Gap_Recall  F1_CV  F1_Test  AUC_CV  AUC_Test
Regresion Logistica A - Todas las variables  0.9704    0.9556  -0.0148     0.9664       0.9510     -0.0153 0.9702   0.9556  0.9941    0.9952
          KNN (k=5) A - Todas las variables  0.9685    0.9556  -0.0130     0.9636       0.9510     -0.0126 0.9684   0.9556  0.9843    0.9900
          SVM (RBF) A - Todas las variables  0.9611    0.9630   0.0019     0.9617       0.9617     -0.0000 0.9613   0.9631  0.9898    0.9908
Regresion Logistica     B - Top 3 variables  0.9648    0.9259  -0.0389     0.9583       0.9085     -0.0498 0.9647   0.9251  0.9912    0.9915
          KNN (k=5)     B - Top 3 variables  0.9667    0.9481  -0.0185     0.9634       0.9404     -0.0230 0.9667   0.9480  0.9823    0.9840
          SVM (RBF)     B - Top 3 variables  0.9574    0.9481  -0.0093     0.9538       0.9404     -0.0134 0.9575   0.94

## 6. Control de fuga de informacion

Reglas aplicadas en este flujo:
- El escalado se realiza dentro de Pipeline.
- Cada fold ajusta scaler/modelo solo con datos de entrenamiento del fold.
- El test set no participa en CV ni en decisiones de seleccion.
- La evaluacion en test se realiza al final para confirmar generalizacion.

Checklist:
- [x] Separacion train/test estratificada conservada.
- [x] CV estratificada para mantener proporcion de clases.
- [x] Condiciones identicas para todos los modelos y escenarios.
- [x] Tabla de resultados por modelo y por escenario.
- [x] Documento breve del metodo de validacion.

## 7. Texto breve del metodo de validacion usado

Se utilizo un esquema de 5-Fold Stratified Cross-Validation sobre el conjunto de entrenamiento heredado de Semana 2 (split 80/20 estratificado con random_state=42). La estratificacion aseguro mantener la proporcion de clases en cada fold, reduciendo sesgos por desbalance. Todos los modelos base (Regresion Logistica, KNN y SVM RBF) fueron evaluados bajo las mismas metricas y con el mismo protocolo, para garantizar comparabilidad. El conjunto de prueba se mantuvo intacto hasta el final para verificar consistencia y evitar fuga de informacion.

## 8. Tuning de hiperparametros (Persona 2): Regresion Logistica y KNN

Objetivo de esta seccion:
- Optimizar LR y KNN con el mismo esquema de validacion de la seccion anterior.
- Usar como metrica principal el recall de la clase maligna (Class=4).
- Repetir el tuning en ambos escenarios de variables para comparacion formal.

In [8]:
# metrica principal para seleccion: recall de clase maligna (Class=4)
scorer_recall_maligna = make_scorer(recall_score, pos_label=4, zero_division=0)

# grilla media para regresion logistica (combinaciones validas por solver)
param_grid_lr = [
    {
        'model__solver': ['lbfgs'],
        'model__penalty': ['l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [2000]
    },
    {
        'model__solver': ['liblinear'],
        'model__penalty': ['l1', 'l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [2000]
    },
    {
        'model__solver': ['saga'],
        'model__penalty': ['l1', 'l2'],
        'model__C': [0.01, 0.1, 1, 3, 10, 30, 100],
        'model__class_weight': [None, 'balanced'],
        'model__max_iter': [3000]
    }
]

# grilla media para knn
param_grid_knn = [
    {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean', 'manhattan']
    },
    {
        'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['minkowski'],
        'model__p': [1, 2]
    }
]

base_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

base_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

print('=' * 90)
print('CONFIGURACION DE TUNING (LR + KNN)')
print('=' * 90)
print('Metrica principal de seleccion: recall_maligna (pos_label=4)')
print(f'CV: StratifiedKFold con {cv_strategy.get_n_splits()} folds')
print(f'Combinaciones LR aproximadas: {sum(len(d["model__C"]) * len(d["model__class_weight"]) * len(d["model__penalty"]) * len(d["model__solver"]) for d in param_grid_lr)}')
print(f'Combinaciones KNN aproximadas: {8*2*2 + 8*2*1*2}')
print('=' * 90)

CONFIGURACION DE TUNING (LR + KNN)
Metrica principal de seleccion: recall_maligna (pos_label=4)
CV: StratifiedKFold con 5 folds
Combinaciones LR aproximadas: 70
Combinaciones KNN aproximadas: 64


In [9]:
tuning_store = {}
rows_best = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    print('\n' + '=' * 90)
    print(f'TUNING EN ESCENARIO: {nombre_escenario}')
    print('=' * 90)

    # LR
    grid_lr = GridSearchCV(
        estimator=base_lr,
        param_grid=param_grid_lr,
        scoring=scorer_recall_maligna,
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )
    grid_lr.fit(X_tr_esc, y_train)

    # KNN
    grid_knn = GridSearchCV(
        estimator=base_knn,
        param_grid=param_grid_knn,
        scoring=scorer_recall_maligna,
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
        return_train_score=True
    )
    grid_knn.fit(X_tr_esc, y_train)

    tuning_store[(nombre_escenario, 'Regresion Logistica')] = grid_lr
    tuning_store[(nombre_escenario, 'KNN')] = grid_knn

    for modelo_nombre, grid_obj in [('Regresion Logistica', grid_lr), ('KNN', grid_knn)]:
        best_idx = grid_obj.best_index_
        best_mean = grid_obj.cv_results_['mean_test_score'][best_idx]
        best_std = grid_obj.cv_results_['std_test_score'][best_idx]

        rows_best.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Best Recall CV (maligna)': round(best_mean, 4),
            'Std Recall CV': round(best_std, 4),
            'Best Params': str(grid_obj.best_params_)
        })

        print(f"{modelo_nombre:22s} | Best Recall CV={best_mean:.4f} +/- {best_std:.4f}")

best_params_df = pd.DataFrame(rows_best)

print('\n' + '=' * 120)
print('TABLA: MEJORES HIPERPARAMETROS (LR + KNN)')
print('=' * 120)
print(best_params_df.to_string(index=False))


TUNING EN ESCENARIO: A - Todas las variables
Regresion Logistica    | Best Recall CV=0.9789 +/- 0.0421
KNN                    | Best Recall CV=0.9632 +/- 0.0488

TUNING EN ESCENARIO: B - Top 3 variables
Regresion Logistica    | Best Recall CV=0.9471 +/- 0.0333
KNN                    | Best Recall CV=0.9578 +/- 0.0268

TABLA: MEJORES HIPERPARAMETROS (LR + KNN)
              Escenario              Modelo  Best Recall CV (maligna)  Std Recall CV                                                                                                                         Best Params
A - Todas las variables Regresion Logistica                    0.9789         0.0421 {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 2000, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
A - Todas las variables                 KNN                    0.9632         0.0488                                                {'model__metric': 'euclidean', 'model__n_neighbors': 9, 'model__weights

In [10]:
def evaluar_en_test(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    if hasattr(modelo, 'predict_proba'):
        y_score = modelo.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    out = {
        'accuracy_test': accuracy_score(y_test, y_pred),
        'precision_maligna_test': precision_score(y_test, y_pred, pos_label=4, zero_division=0),
        'recall_maligna_test': recall_score(y_test, y_pred, pos_label=4, zero_division=0),
        'f1_maligna_test': f1_score(y_test, y_pred, pos_label=4, zero_division=0),
        'fn_test': int(((y_test == 4) & (y_pred != 4)).sum())
    }

    if y_score is not None:
        out['auc_test'] = roc_auc_score(y_test, y_score, labels=LABELS_CLASE)
    else:
        out['auc_test'] = np.nan

    return out

rows_compare = []
rows_fit_diag = []

for nombre_escenario, (X_tr_esc, X_te_esc) in escenarios.items():
    for modelo_nombre in ['Regresion Logistica', 'KNN']:
        base_model = modelos_base['Regresion Logistica'] if modelo_nombre == 'Regresion Logistica' else modelos_base['KNN (k=5)']
        tuned_grid = tuning_store[(nombre_escenario, modelo_nombre)]
        tuned_model = tuned_grid.best_estimator_

        # base metrics (CV heredado + test)
        base_key = f"{'Regresion Logistica' if modelo_nombre == 'Regresion Logistica' else 'KNN (k=5)'} | {nombre_escenario}"
        base_cv = cv_results_store[base_key]
        base_cv_recall = base_cv['test_recall_macro'].mean()
        base_test = evaluar_en_test(base_model.fit(X_tr_esc, y_train), X_te_esc, y_test)

        # tuned metrics (CV principal + test)
        tuned_best_idx = tuned_grid.best_index_
        tuned_cv_recall = tuned_grid.cv_results_['mean_test_score'][tuned_best_idx]
        tuned_cv_train_recall = tuned_grid.cv_results_['mean_train_score'][tuned_best_idx]
        tuned_gap = tuned_cv_train_recall - tuned_cv_recall
        tuned_test = evaluar_en_test(tuned_model, X_te_esc, y_test)

        rows_compare.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Tipo': 'Base',
            'Recall_CV_objetivo': round(base_cv_recall, 4),
            'Recall_Test_maligna': round(base_test['recall_maligna_test'], 4),
            'F1_Test_maligna': round(base_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(base_test['accuracy_test'], 4),
            'AUC_Test': round(base_test['auc_test'], 4),
            'FN_Test': base_test['fn_test']
        })

        rows_compare.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Tipo': 'Optimizado',
            'Recall_CV_objetivo': round(tuned_cv_recall, 4),
            'Recall_Test_maligna': round(tuned_test['recall_maligna_test'], 4),
            'F1_Test_maligna': round(tuned_test['f1_maligna_test'], 4),
            'Accuracy_Test': round(tuned_test['accuracy_test'], 4),
            'AUC_Test': round(tuned_test['auc_test'], 4),
            'FN_Test': tuned_test['fn_test']
        })

        rows_fit_diag.append({
            'Escenario': nombre_escenario,
            'Modelo': modelo_nombre,
            'Recall_train_CV_best': round(tuned_cv_train_recall, 4),
            'Recall_valid_CV_best': round(tuned_cv_recall, 4),
            'Gap_train_valid': round(tuned_gap, 4),
            'Diagnostico_ajuste': 'posible sobreajuste' if tuned_gap > 0.05 else ('posible subajuste' if tuned_cv_recall < 0.85 else 'estable')
        })

df_compare = pd.DataFrame(rows_compare)
df_fit_diag = pd.DataFrame(rows_fit_diag)

# deltas base vs optimizado
rows_delta = []
for (esc, mod), grp in df_compare.groupby(['Escenario', 'Modelo']):
    base_row = grp[grp['Tipo'] == 'Base'].iloc[0]
    opt_row = grp[grp['Tipo'] == 'Optimizado'].iloc[0]
    rows_delta.append({
        'Escenario': esc,
        'Modelo': mod,
        'Delta_Recall_CV_objetivo': round(opt_row['Recall_CV_objetivo'] - base_row['Recall_CV_objetivo'], 4),
        'Delta_Recall_Test_maligna': round(opt_row['Recall_Test_maligna'] - base_row['Recall_Test_maligna'], 4),
        'Delta_F1_Test_maligna': round(opt_row['F1_Test_maligna'] - base_row['F1_Test_maligna'], 4),
        'Delta_Accuracy_Test': round(opt_row['Accuracy_Test'] - base_row['Accuracy_Test'], 4),
        'Delta_FN_Test': int(opt_row['FN_Test'] - base_row['FN_Test'])
    })

df_delta = pd.DataFrame(rows_delta)

print('=' * 140)
print('TABLA: COMPARACION BASE VS OPTIMIZADO (LR + KNN)')
print('=' * 140)
print(df_compare.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA: DELTAS DE MEJORA (OPTIMIZADO - BASE)')
print('=' * 140)
print(df_delta.to_string(index=False))

print('\n' + '=' * 140)
print('TABLA: ANALISIS DE AJUSTE (SEÑALES DE SOBREAJUSTE/SUBAJUSTE)')
print('=' * 140)
print(df_fit_diag.to_string(index=False))

TABLA: COMPARACION BASE VS OPTIMIZADO (LR + KNN)
              Escenario              Modelo       Tipo  Recall_CV_objetivo  Recall_Test_maligna  F1_Test_maligna  Accuracy_Test  AUC_Test  FN_Test
A - Todas las variables Regresion Logistica       Base              0.9664               0.9362           0.9362         0.9556    0.9952        3
A - Todas las variables Regresion Logistica Optimizado              0.9789               0.9574           0.9474         0.9630    0.9944        2
A - Todas las variables                 KNN       Base              0.9636               0.9362           0.9362         0.9556    0.9900        3
A - Todas las variables                 KNN Optimizado              0.9632               0.9149           0.9247         0.9481    0.9889        4
    B - Top 3 variables Regresion Logistica       Base              0.9583               0.8511           0.8889         0.9259    0.9915        7
    B - Top 3 variables Regresion Logistica Optimizado              0

## 9. Resumen tecnico de tuning (Persona 2)

- Se optimizaron Regresion Logistica y KNN en ambos escenarios de variables usando GridSearchCV.
- La metrica principal para seleccion fue el recall de la clase maligna (Class=4), priorizando reduccion de falsos negativos.
- La comparacion base vs optimizado se reporta en CV y en test para confirmar si la mejora se sostiene fuera de validacion.
- El analisis de ajuste se basa en el gap entre recall de entrenamiento y validacion en CV para detectar señales de sobreajuste/subajuste.

In [11]:
# resumen compacto de resultados de tuning (salida corta para lectura rapida)
print('=' * 100)
print('RESUMEN COMPACTO | MEJORES PARAMETROS')
print('=' * 100)
print(best_params_df[['Escenario', 'Modelo', 'Best Recall CV (maligna)', 'Std Recall CV']].to_string(index=False))

print('\n' + '=' * 100)
print('RESUMEN COMPACTO | DELTAS OPTIMIZADO - BASE')
print('=' * 100)
print(df_delta.to_string(index=False))

print('\n' + '=' * 100)
print('RESUMEN COMPACTO | DIAGNOSTICO DE AJUSTE')
print('=' * 100)
print(df_fit_diag.to_string(index=False))

RESUMEN COMPACTO | MEJORES PARAMETROS
              Escenario              Modelo  Best Recall CV (maligna)  Std Recall CV
A - Todas las variables Regresion Logistica                    0.9789         0.0421
A - Todas las variables                 KNN                    0.9632         0.0488
    B - Top 3 variables Regresion Logistica                    0.9471         0.0333
    B - Top 3 variables                 KNN                    0.9578         0.0268

RESUMEN COMPACTO | DELTAS OPTIMIZADO - BASE
              Escenario              Modelo  Delta_Recall_CV_objetivo  Delta_Recall_Test_maligna  Delta_F1_Test_maligna  Delta_Accuracy_Test  Delta_FN_Test
A - Todas las variables                 KNN                   -0.0004                    -0.0213                -0.0115              -0.0075              1
A - Todas las variables Regresion Logistica                    0.0125                     0.0212                 0.0112               0.0074             -1
    B - Top 3 variables 

## 10. Texto base para documento (Semana 3)

En esta etapa se optimizaron los modelos de Regresion Logistica y KNN bajo el mismo protocolo experimental definido para la semana: split 80/20 estratificado heredado de Semana 2 y validacion cruzada estratificada de 5 folds sobre el conjunto de entrenamiento. La seleccion de hiperparametros se realizo con GridSearchCV, priorizando como metrica principal el recall de la clase maligna (Class = 4), debido al impacto clinico de los falsos negativos.

Se ejecuto tuning en ambos escenarios de variables (A: todas las variables, B: top 3 variables), manteniendo condiciones comparables para evitar sesgos de evaluacion. Para cada modelo se reportaron los mejores hiperparametros y el desempeno en validacion, y posteriormente se contrasto su comportamiento en el conjunto de prueba para verificar si la mejora se sostenia fuera de la validacion.

Los resultados muestran que la Regresion Logistica optimizada mejora de forma consistente en el escenario completo (A), con incremento en recall de maligna en test y reduccion de falsos negativos. En el escenario reducido (B), la Regresion Logistica tambien mejora en test, aunque con menor estabilidad en validacion. En KNN, la optimizacion no produjo ganancias relevantes; en el escenario completo se observa una ligera degradacion y en el escenario top 3 el desempeno se mantiene practicamente igual.

El analisis del gap entre desempeno de entrenamiento y validacion para las configuraciones optimas no evidencia sobreajuste fuerte en los modelos ajustados de LR y KNN (gaps pequenos y comportamiento estable). Por tanto, para la comparacion final de Semana 3, el candidato de Regresion Logistica optimizada queda validado como alternativa competitiva, mientras que KNN optimizado se considera sin mejora significativa respecto a su baseline.